# Chapter 4 &mdash; The Pumping Lemma: Statement and Pigeonhole

**Concept 14 of the Chapter 4 decomposition:** *The Pumping Lemma for Regular Languages: Statement, Proof Sketch, and Pigeonhole*

If $L$ is regular then long strings split as $xyz$ with $|xy|\le N$, $y\ne\varepsilon$, and $xy^iz\in L$ for all $i$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Pumping-Lemma-Statement/Concept-Pumping-Lemma-Statement.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


> **IF** $L$ is regular, **THEN** there is $N$ such that for any $w\in L$ with
> $|w|\ge N$, we can write $w = xyz$ where $y$ is non-empty, $|xy| \le N$, and
> **for all $i \ge 0$, $xy^iz \in L$**.

**Why a repeat must occur:** the **pigeonhole principle**. With $M \ge N$ transitions
there must be a repeated state &mdash; states are like duck digits and transitions the
webs between them, so an $N$-state DFA admits at most a journey of length $N-1$
without repeating.

**Practical advice: pick $y$ first.** Then $x$ is whatever precedes it and $z$ is the
rest.

## 2. Definitions

### Pigeonhole, demonstrated

In [ ]:
def must_repeat(D, s):
    """Any run of length >= |Q| revisits a state."""
    q, seen = D["q0"], [D["q0"]]
    for ch in s:
        q = step_dfa(D, q, ch); seen.append(q)
    return len(seen) > len(set(seen)), len(seen), len(set(seen))

### The lemma as a checkable predicate

In [ ]:
D = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')
N = len(D["Q"])

def pumping_splits(w, N):
    """All splits with y non-empty and |xy| <= N."""
    return [(w[:i], w[i:j], w[j:])
            for i in range(N+1) for j in range(i+1, min(N, len(w))+1)]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;13.&nbsp;Visitation Numbers, Pumping Up and Pumping Down](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Pumping-Up-And-Down/Concept-Pumping-Up-And-Down.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;15.&nbsp;One-Way Implication: Use the Pumping Lemma Only to Disprove Regularity](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-One-Way-Implication/Concept-One-Way-Implication.ipynb)&nbsp;&rarr;

---

## 3. Tests

Pigeonhole: a long enough run cannot avoid a repeat.

In [ ]:
for w in ['01', '010', '0100', '010010']:
    rep, visits, distinct = must_repeat(D, w)
    print("|w|=%d  visits=%d distinct=%d  repeated? %s" % (len(w), visits, distinct, rep))
assert must_repeat(D, '0100')[0]
print("\n|Q| = %d, so any run of %d+ states must reuse one." % (N, N+1))

For a string in the language, **some** split pumps &mdash; the lemma promises at least one.

In [ ]:
w = '0100'
good = [(x,y,z) for (x,y,z) in pumping_splits(w, N)
        if y and all(accepts_dfa(D, x + y*i + z) for i in range(6))]
print("w =", w, " splits satisfying the lemma:")
for x,y,z in good: print("   x=%-4r y=%-4r z=%-4r" % (x,y,z))
assert good, "the lemma guarantees at least one pumping split"

Note the direction: the lemma says **some** split works, not **every** one.

In [ ]:
bad = [(x,y,z) for (x,y,z) in pumping_splits(w, N)
       if y and not all(accepts_dfa(D, x + y*i + z) for i in range(6))]
print("splits that do NOT pump :", len(bad))
print("\nThat is fine -- the lemma is existential over splits when PROVING regularity,")
print("which is exactly why refuting it needs ALL splits (Concept 17).")

## 4. Exercises


1. State the pigeonhole principle in one sentence, then apply it to a 5-state DFA.
2. Why must $|xy| \le N$? Which choice in the proof forces it?
3. Pick $y$ first for the string `010010`. Do $x$ and $z$ follow automatically?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-Pumping-Lemma-Statement')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')